## Text Summarizer Plugin

### Introduction
This notebook demonstrates how a browser plugin, built using Flask, leverages an OpenVINO™ backend to efficiently summarize any webpage via a URL or any PDF via an upload. The plugin utilizes Langchain tools for tasks such as text splitting and managing a vectorstore.

### Pre-requisites

#### Install the below necessary tools/packages:
   - [Git on Windows](https://git-scm.com/downloads)
   - [Miniforge](https://conda-forge.org/download/)
   - [Google Chrome for Windows](https://www.google.com/chrome/?brand=OZZY&ds_kid=43700080794581137&gad_source=1&gclid=Cj0KCQiAoae5BhCNARIsADVLzZdwNNB5nIyjZ8OyCzg6h_cCig1eoaYquUSEd7BAigJhTzps1Kxuop8aArE6EALw_wcB&gclsrc=aw.ds)


#### Clone the Repository

In [ ]:
! git clone -b OpenVINO-backend https://github.com/AlekhyaVemuri/Text-Summarizer.git

#### Conda Environment Creation

In [ ]:
! conda create -n summarizer_plugin python=3.11 libuv

In [ ]:
! conda activate summarizer_plugin

#### Installing Dependencies

In [ ]:
! cd Text-Summarizer

In [ ]:
! pip install -r requirements.txt

#### Download and Convert the Huggingface Model to OpenVINO IR Format:

##### Login to Huggingface:
Generate a token from Huggingface for private/gated models like Meta Llama, etc. To access such private/gated models, refer to [Huggingface documentation](https://huggingface.co/docs/hub/en/models-gated).

In [1]:
from huggingface_hub import login
login()

##### Converting a huggingface model to OpenVINO
Convert the models using `optimum-cli`

In [ ]:
! mkdir models && cd models     

In [ ]:
! optimum-cli export openvino --model Qwen/Qwen2-7B-Instruct --weight-format int4 ov_qwen7b

In [ ]:
! optimum-cli export openvino --model meta-llama/Llama-2-7b-chat-hf --weight-format int4 ov_llama_2

>**Note**: [Raise access request](https://www.llama.com/llama-downloads) for Llama models as it is a gated repository.


#### Load the extension

To load an unpacked extension in developer mode:
- Go to the Extensions page by entering **chrome://extensions** in a new tab. (By design chrome:// URLs are not linkable.)
    - Alternatively, **click the Extensions menu puzzle button and select Manage Extensions** at the bottom of the menu.
    - Or, click the Chrome menu, hover over More Tools, then select Extensions.
- Enable **Developer Mode** by clicking the toggle switch next to Developer mode.
- Click the **Load unpacked** button and select the extension directory.
- Refer to [Chrome’s development documentation](https://developer.chrome.com/docs/extensions/get-started/tutorial/hello-world#load-unpacked) for further details.

<img src="./../assets/img1.png" width=250 height=250 >







#### Pin the extension
Pin your extension to the toolbar to quickly access your extension.

<img src="./../assets/img2.png" height=250 width=250>


### Code Sample Structure
In this browser plugin we have divided into two parts 
- **Backend** - In the backend, we have two python files `code.py` and `server.py`
  - `code.py` manages data pre-processing tasks
  - `server.py` manages flask server-side operations
- **Extension** - In the extension we have the front end code required for the browser plugin (popup.html, popup.js, style.css, manifest.json)


### Backend code for Text Summarization 

#### Importing the necessary libraries

In [ ]:
from transformers import AutoTokenizer, pipeline
from optimum.intel import OVModelForCausalLM
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader

#### Prompt Templates for Summarization & Question Answering Bot
Here we have created two variables for prompt template so that it can be called later on , one template for summarization and one for query asked in the bot

In [ ]:
#prompt template for summarization
summary_template= """Write a concise summary of the following: "{context}" CONCISE SUMMARY: """
#prompt template for query
query_template="""Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    Use 10 words maximum and keep the answer as concise as possible in one sentence.
    Always say "thanks for asking!" at the end of the answer.
 
    {context}
 
    Question: {question}
 
    Helpful Answer:"""

#### Preprocessing 

* Loads page content from the webpage/PDF. Document loaders in RAG are used to load and preprocess the documents that will be used for retrieval     during the question answering process.
* Splits the page data using Recursive Character Text Splitter & creates embeddings using HuggingFace Embeddings. RecursiveCharacterTextSplitter is used to split text into smaller pieces recursively at the character level.
* In RAG, embeddings plays a crucial role in retrieval of relevant documents for a given query and Sentence Transformers helps to generate embeddings for each document in your knowledge base.
* This is further stored into ChromaDB for futher retrieval usage .Chroma is a vector store and embeddings database designed from the ground-up to make it easy to build AI applications with embeddings.

In [ ]:
#Function created for preprocessing 
def pre_processing(loader):    
    try:
        page_data = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
        all_splits = text_splitter.split_documents(page_data)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        global vectorstore
        vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)  
        return vectorstore
    except Exception as e:
        print(f"Error while processing Webpage/PDF page content: {e}")

#### Loading LLM models
In this we are trying to create a common function for loading the LLM models in a drop-down and then trying to return that LLM model which will be used later on for summarization

In [ ]:
#function created for laoding the LLM
def load_llm(model_id):
    if model_id:
        try:
            if model_id=="Meta LLama 2":
                model_path=r"..\models\ov_llama_2"
            elif model_id=="Qwen 7B Instruct":
                model_path=r"..\models\ov_qwen7b"
            model = OVModelForCausalLM.from_pretrained(model_path , device='GPU')
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            pipe=pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=4000,  
                device=model.device
            )
            global llm_model 
            llm_model = HuggingFacePipeline(pipeline=pipe)
            return llm_model
        except Exception as e:
            print(f"Failed to load the model. Please check whether the model_path is correct. \n Error: {e}")

####  URL Summarization
Here we try to load the web page when a user enters an URL into the plugin which in return loads the page data and passes into the RetrievalQA chain. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it . Here we are using WebBaseLoader to load the documents from the web.
* The **WebBaseLoader** in Retrieval Augmented Generation (RAG) is a type of document loader that is designed to load documents from the web.The WebBaseLoader is used when the documents for retrieval are not stored locally or in a Hugging Face dataset, but are instead located on the web.
* **RetrievalQA** is a type of question answering system that uses a retriever to fetch relevant documents given a question, and then uses a reader to extract the answer from the retrieved documents.

In [ ]:
#function created for URL content summarization
def web_out(urls):
    try:
        loader = WebBaseLoader(urls)
        global summ_vectorstore 
        summ_vectorstore = pre_processing(loader)
        prompt = PromptTemplate(
            template=summary_template,
            input_variables=["context", "question"]
        )
    
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm_model,
            retriever=summ_vectorstore.as_retriever(),
            chain_type="stuff",
            chain_type_kwargs={"prompt": prompt},
            return_source_documents=False,
        )
        
        question = "Please summarize the context in one paragraph of 100 words"
        summary = qa_chain({'query': question})
        response = summary['result']
        summary_start = response.find("CONCISE SUMMARY:")
        concise_summary = response[summary_start + len("CONCISE SUMMARY:"):].strip()
        return concise_summary
    except Exception as e:
        print(f"Failed to summarize webpage \n Error: {e}")

#### URL Question Answering BOT
The function defined below does a follow up questions to the bot related to the content being uploaded as URL post summarization. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it .
Here we are taking the **query template** which is declared as global and created a chain with llm model, retreiver and chain type as "stuff" and then based on the query asked by the user we get an answer from the LLM model.

In [ ]:
#function created for QnA bot for URL
def url_query(query):
    try:
        prompt = PromptTemplate(
            template=query_template,
            input_variables=["context", "question"]
            )
        reduce_chain = RetrievalQA.from_chain_type(
                llm=llm_model,
                retriever=summ_vectorstore.as_retriever(),
                chain_type="stuff",
                chain_type_kwargs={"prompt": prompt},
                return_source_documents=False
            )
        summary = reduce_chain({'query': query})
        summ_vectorstore.delete
        response = summary['result']
        summary_start = response.find("Helpful Answer:")
        concise_summary = response[summary_start + len("Helpful Answer:"):].strip()
        return concise_summary
    except Exception as e:
        print(f"Error in Webpage Summarizer QA BoT: {e}")


#### PDF Summarization
Here we try to take a PDF file as input into the plugin which loads the page data and passes into the RetrievalQA chain. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it . Here we are using PyPDF loader to load the PDF document.
* **PyPDFLoader** is a document loader within the LangChain framework specifically designed to handle PDF files. It allows you to extract text from PDF documents and load them into a format suitable for language models and other text-based applications.


In [ ]:
#function created for PDF summarization
def pdf_out(pdf):
    try:
        loader = PyPDFLoader(pdf, extract_images=False)
        global pdf_vectorstore
        pdf_vectorstore=pre_processing(loader)
    
        prompt = PromptTemplate(
            template=summary_template,
            input_variables=["context", "question"]
        )
        reduce_chain = RetrievalQA.from_chain_type(
            llm=llm_model,
            retriever=pdf_vectorstore.as_retriever(),
            chain_type="stuff",
            chain_type_kwargs={"prompt": prompt},
            return_source_documents=False,
        )
        question = "Please summarize the context in one paragraph of 60 words"
        summary = reduce_chain({'query': question})

        response = summary['result']
        summary_start = response.find("CONCISE SUMMARY:")
        concise_summary = response[summary_start + len("CONCISE SUMMARY:"):].strip()
        return concise_summary
    except Exception as e:
        print(f"Failed to summarize PDF \n Error: {e}")

#### PDF Question Answering BOT
The function defined below does a follow up questions to the bot related to the content being uploaded as PDF post summarization. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it . Here we are taking the **query template** which is declared as global and created a chain with llm model, retreiver and chain type as "stuff" and then based on the query asked by the user we get an answer from the LLM model.

In [ ]:
#function created for Question Answering bot for PDF
def pdf_query(query):
    try:
        prompt = PromptTemplate(
            template=query_template,
            input_variables=["context", "question"]
            )
        reduce_chain = RetrievalQA.from_chain_type(
                llm=llm_model,
                retriever=pdf_vectorstore.as_retriever(),
                chain_type="stuff",
                chain_type_kwargs={"prompt": prompt},
                return_source_documents=False
            )
        summary = reduce_chain({'query': query})
        response = summary['result']
        summary_start = response.find("Helpful Answer:")
        concise_summary = response[summary_start + len("Helpful Answer:"):].strip()
        print(concise_summary)

In [ ]:
! python server.py